# DeepSeek-V4: Indexer

The Indexer selects top-k compressed KV blocks via learned multi-head scoring.
This notebook demonstrates:
1. Simplified Indexer (one-pass top-k selection)
2. V4 Indexer (multi-head scoring with weight projection)
3. Causal mask and index offset
4. Overlap mechanism in CSA Compressor

# Configuration

In [1]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42)

In [2]:
from dataclasses import dataclass
from typing import Tuple, Optional, Literal

@dataclass
class ModelArgs:
    max_batch_size: int = 4
    max_seq_len: int = 4096
    vocab_size: int = 129280
    dim: int = 4096
    moe_inter_dim: int = 4096
    n_layers: int = 7
    n_heads: int = 64
    
    # mqa
    q_lora_rank: int = 1024
    head_dim: int = 512
    rope_head_dim: int = 64
    norm_eps: float = 1e-6
    o_groups: int = 8
    o_lora_rank: int = 1024
    window_size: int = 128
    compress_ratios: Tuple[int] = (0, 0, 4, 128, 4, 128, 4, 0)
    
    # indexer
    index_n_heads: int = 64
    index_head_dim: int = 128
    index_topk: int = 512

args = ModelArgs()
args.index_topk = 3

# Input data
bsz = 2
seq_len = 9
X = torch.randn(bsz, seq_len, args.dim)

## 1. Simplified Indexer: One-Pass Top-K Selection

The simplest form: project Q/K and select top-k indices in one pass.

One-pass: compute dot-product scores between Q and K, then take top-k per query.

In [3]:
def attn(Q, K, V):
    return Q @ K.transpose(1, 2) @ V

def SimpleIndexer(Q, K, index_topk):
    """Simplified: compute Q@K^T scores and take top-k per query."""
    S = Q @ K.transpose(1, 2)
    topk_ids = S.topk(index_topk, dim=-1)[1]
    return topk_ids

topk_idxs = SimpleIndexer(Q=X, K=X, index_topk=args.index_topk)

In [4]:
def AttentionWithIndexer(X, topk_idxs):
    """Per-query sparse attention: each query attends only to its top-k KV."""
    B, L, D = X.shape
    O = torch.zeros_like(X)
    for i in range(B):
        for j in range(L):
            idxs = topk_idxs[i, [j]]
            Q, K, V = X[i, [j], :], X[i, idxs], X[i, idxs]
            O[i, j] = attn(Q, K, V)
        break  # demo: first batch only
    return O

O = AttentionWithIndexer(X, topk_idxs)
print(f"Input: {list(X.shape)}, Output: {list(O.shape)}")
print(f"Each query selects {args.index_topk} KV positions via Indexer")

Input: [2, 9, 4096], Output: [2, 9, 4096]
Each query selects 3 KV positions via Indexer


## 2. V4 Indexer: Multi-Head Scoring with Weight Projection

The V4 Indexer has two key differences from the simplified version:
1. **Multi-head scoring**: `q` has `index_n_heads=64` heads, `kv` is single-head
2. **Weight projection**: a learned `weights_proj` produces per-head importance weights

The scoring process:
- `index_score = einsum("bshd,btd->bsht", q, kv_cache)` — multi-head dot product
- `weight_score = (index_score.relu() * weights.unsqueeze(-1)).sum(dim=2)` — weighted sum across heads
- `topk_idxs = weight_score.topk(k)` — final top-k selection

In [5]:
class IndexerAlgorithm(torch.nn.Module):
    """V4 Indexer: multi-head scoring with learned weight projection."""
    def __init__(self, args: ModelArgs, compress_ratio: int = 4):
        super().__init__()
        self.n_heads = args.index_n_heads
        self.head_dim = args.index_head_dim
        self.index_topk = args.index_topk
        self.q_lora_rank = args.q_lora_rank
        self.dim = args.dim
        
        self.wq_b = nn.Linear(self.q_lora_rank, self.n_heads * self.head_dim)
        self.weights_proj = nn.Linear(self.dim, self.n_heads)
        self.softmax_scale = self.head_dim ** -0.5

    def forward(self, x, qr, kv):
        # Step 1: Project query to multi-head
        q = self.wq_b(qr)
        q = q.unflatten(-1, (self.n_heads, self.head_dim))
        print(f"q shape after unflatten: {q.shape}")  # [B, S, H, D]

        B, S, H, D = q.shape
        _, T, _ = kv.shape
        q = q.transpose(1, 2)      # [B, H, S, D]
        kv = kv.transpose(1, 2).unsqueeze(dim=1)  # [B, 1, T, D]

        # Step 2: Multi-head dot product scoring
        index_score = q @ kv         # [B, H, S, T]
        index_score = index_score.transpose(1, 2)  # [B, S, H, T]

        # Step 3: Learned weight projection and aggregation
        weights = self.weights_proj(x) * (self.softmax_scale * self.n_heads ** -0.5)
        weight_score = index_score.relu_() * weights.unsqueeze(-1)  # [B, S, H, T]
        index_score = weight_score.sum(dim=2)  # [B, S, T] — aggregate across heads

        # Step 4: Top-k selection
        print(index_score.shape)
        _, topk_idxs = index_score.topk(self.index_topk, dim=-1)
        return topk_idxs

indexer = IndexerAlgorithm(args)

In [6]:
print(indexer)  # Show parameter structure

IndexerAlgorithm(
  (wq_b): Linear(in_features=1024, out_features=8192, bias=True)
  (weights_proj): Linear(in_features=4096, out_features=64, bias=True)
)


In [10]:
# Run IndexerAlgorithm with uncompressed and compressed KV
seq_len = 13

X = torch.randn(bsz, seq_len, args.dim)
Q = torch.randn(bsz, seq_len, args.q_lora_rank)
KV = torch.randn(bsz, seq_len, 2 * args.dim // args.n_heads)

# With full-length KV (expensive)
topk_idxs_full = indexer(X, Q, KV)
print(f"Topk indices (full KV):\n{topk_idxs_full}")

# With compressed KV (efficient)
cKV = torch.randn(bsz, seq_len // args.compress_ratios[2], 2 * args.dim // args.n_heads)
topk_idxs_comp = indexer(X, Q, cKV)
print(f"\nTopk indices (compressed KV, len={seq_len // 4}):\n{topk_idxs_comp}")
print(f"Full KV length: {seq_len}, Compressed KV length: {seq_len // 4}")

q shape after unflatten: torch.Size([2, 13, 64, 128])
torch.Size([2, 13, 13])
Topk indices (full KV):
tensor([[[12, 10,  6],
         [10,  6,  2],
         [ 9,  4, 10],
         [ 0, 12,  4],
         [ 9,  7,  2],
         [ 6,  0, 12],
         [ 5,  6,  8],
         [ 8,  1,  5],
         [ 9,  5,  2],
         [ 3,  0,  4],
         [ 9,  6,  0],
         [ 4,  5, 11],
         [ 3, 10,  0]],

        [[ 7,  3,  6],
         [ 3,  4,  2],
         [12, 11,  3],
         [12, 11,  1],
         [ 3,  7,  0],
         [ 0,  8, 11],
         [11,  9,  1],
         [ 9,  4,  8],
         [11,  7,  8],
         [ 7,  6,  2],
         [11,  3,  4],
         [ 5,  8,  3],
         [ 8,  7, 10]]])
q shape after unflatten: torch.Size([2, 13, 64, 128])
torch.Size([2, 13, 3])

Topk indices (compressed KV, len=3):
tensor([[[0, 1, 2],
         [2, 0, 1],
         [1, 0, 2],
         [2, 1, 0],
         [1, 2, 0],
         [0, 1, 2],
         [1, 0, 2],
         [2, 0, 1],
         [2, 1, 0],
 

### FLOPs Reduction via Compressed KV

The Indexer reduces computation through three dimensions:
1. **Fewer heads**: `index_n_heads=64` (vs 128 for attention)
2. **Smaller head_dim**: `index_head_dim=128` (vs 512 for attention)
3. **Shorter KV**: compressed from `L` to `L/ratio` (e.g., 1M -> 250K)

In V4, the Indexer specifically uses compressed KV (method 3) for scoring.

In [11]:
# Demonstrate compression ratio effect on KV length
seq_len = 9
ratio = 4
ckv_len = seq_len // ratio
reminder = seq_len - ckv_len * ratio
print(f"seq_len={seq_len}, ratio={ratio}: compressed_len={ckv_len}, remainder={reminder}")
print(f"Compression: {seq_len} tokens -> {ckv_len} compressed blocks")

seq_len=9, ratio=4: compressed_len=2, remainder=1
Compression: 9 tokens -> 2 compressed blocks


In [12]:
# Per-query causal visibility: how many compressed KV blocks can query j see?
for i in range(seq_len):
    print(f"query {i}: can see {(i+1) // ratio} compressed KV block(s)")

query 0: can see 0 compressed KV block(s)
query 1: can see 0 compressed KV block(s)
query 2: can see 0 compressed KV block(s)
query 3: can see 1 compressed KV block(s)
query 4: can see 1 compressed KV block(s)
query 5: can see 1 compressed KV block(s)
query 6: can see 1 compressed KV block(s)
query 7: can see 2 compressed KV block(s)
query 8: can see 2 compressed KV block(s)


## 3. Indexer Score with Weight Projection

The final scoring combines two components:
1. **`weights`**: projected from input `x` via `weights_proj`, shape `[B, S, H]`
2. **`index_score`**: multi-head dot product `q @ kv_cache`, shape `[B, S, H, T]`

Final aggregation:
```python
index_score = einsum("bshd,btd->bsht", q, kv_cache[:bsz, :end_pos // ratio])
weight_score = (index_score.relu() * weights.unsqueeze(-1)).sum(dim=2)  # sum across heads -> [B, S, T]
```

## 4. Causal Mask and Index Offset

The Indexer applies causal masking to ensure each query only sees past compressed KV blocks.
Three steps:
1. Apply causal mask (add -inf for future positions)
2. Select top-k from valid positions
3. Add offset (window_size) and set invalid positions to -1

### Causal Mask Demo

For `seq_len=7, ratio=2, topk=3, offset=100`:
- Query 0-2: no compressed KV visible (all masked)
- Query 3-6: can see compressed blocks `[0, ...]`
- Valid indices are offset by `window_size=100` to point into the kv_cache

In [13]:
def IndexerMask(score, L=7, k=3, r=2, offset=128):
    """Apply causal mask, top-k selection, and offset to Indexer scores.
    
    1. Causal mask: block j can only see blocks < (j+1)//r
    2. Top-k: select k highest-score positions per query
    3. Offset: add window_size offset; set invalid positions to -1
    """
    B, L_seq, _ = score.shape
    ckv_len = L_seq // 2

    # Step 1: Causal compressed-KV mask
    mask = torch.zeros(B, L_seq, ckv_len)
    for i in range(B):
        for j in range(L_seq):
            s = (j + 1) // r  # how many blocks query j can see
            if s < ckv_len:
                mask[i, j, s:] = float('-inf')
    score = score + mask

    # Step 2: Top-k selection
    topk_idxs = score.topk(k, dim=-1)[1]

    # Step 3: Add offset and mask invalid positions
    for i in range(B):
        for j in range(L_seq):
            s = (j + 1) // r
            topk_idxs[i, j, :s] += offset
            topk_idxs[i, j, s:] = -1  # -1 means ignore
    return topk_idxs

B = 2; L = 7; r = 2
score = torch.randn(B, L, L // r)
topk_idxs = IndexerMask(score, L, k=3, r=2, offset=100)
print(f"Indexer output (seq_len={L}, ratio={r}, topk=3, offset=100):")
print(topk_idxs)
print(f"\nInterpretation:")
print(f"  -1 means 'ignore' (no valid compressed KV)")
print(f"  100+ means 'compressed KV block at index (value-100)'")

Indexer output (seq_len=7, ratio=2, topk=3, offset=100):
tensor([[[ -1,  -1,  -1],
         [100,  -1,  -1],
         [100,  -1,  -1],
         [100, 101,  -1],
         [101, 100,  -1],
         [100, 102, 101],
         [102, 101, 100]],

        [[ -1,  -1,  -1],
         [100,  -1,  -1],
         [100,  -1,  -1],
         [100, 101,  -1],
         [101, 100,  -1],
         [102, 101, 100],
         [102, 101, 100]]])

Interpretation:
  -1 means 'ignore' (no valid compressed KV)
  100+ means 'compressed KV block at index (value-100)'


## 5. Overlap Mechanism

When `compress_ratio=4` (CSA), the Compressor enables overlap to smooth block boundaries.

The overlap mechanism:
1. Projection output dimension doubles: `coff = 1 + overlap = 2`
2. Before compression, adjacent blocks are concatenated for continuity
3. `overlap_transform` interleaves current and previous block data

```python
class Compressor(nn.Module):
    def __init__(self, args, compress_ratio=4, head_dim=512, rotate=False):
        self.overlap = compress_ratio == 4
        coff = 1 + self.overlap  # coff = 2 when overlap is True
        self.wkv = Linear(dim, coff * head_dim, dtype=torch.float32)
        self.wgate = Linear(dim, coff * head_dim, dtype=torch.float32)
        self.ape = nn.Parameter(torch.empty(compress_ratio, coff * head_dim))
```

In [14]:
def overlap_transform(x, value=0.0):
    """Transform [b, s, r, 2d] -> [b, s, 2r, d] by interleaving current and previous blocks.
    
    For each position s:
    - Right half [s, r:] = current block right part (x[s, :, d:])
    - Left half [s, :r] shifted from previous position (x[s-1, :, :d])
    This creates overlap: block s incorporates information from block s-1.
    """
    b, s, r, D = x.shape  # r=compress_ratio, D=2*head_dim when overlap
    d = D // 2
    new_x = x.new_full((b, s, 2 * r, d), value)
    new_x[:, :, r:] = x[:, :, :, d:]   # right half: current block
    new_x[:, 1:, :r] = x[:, :-1, :, :d] # left half: previous block (shifted)
    return new_x

# Demonstrate overlap transform
x = torch.randn(1, 3, 3, 2)  # [b=1, s=3, r=3, 2d=2]
print(f"Input shape: {x.shape}, values:")
print(x[0])

y = overlap_transform(x)
print(f"\nOutput shape: {y.shape}")
print(f"Overlap effect: each position now has 2r={2*3} entries of size d={2//2}")
print(f"Block 0: padded left half (value=0) + current right half")
print(f"Block 1+: previous left half + current right half")
print(y[0])

Input shape: torch.Size([1, 3, 3, 2]), values:
tensor([[[ 2.4564,  0.9430],
         [-0.9331,  0.9760],
         [ 1.8371,  0.2182]],

        [[ 0.6972, -0.4916],
         [ 1.7593, -1.1458],
         [ 0.0283, -1.0832]],

        [[ 1.3869,  0.0424],
         [-1.3290,  0.1389],
         [ 1.1720,  0.6271]]])

Output shape: torch.Size([1, 3, 6, 1])
Overlap effect: each position now has 2r=6 entries of size d=1
Block 0: padded left half (value=0) + current right half
Block 1+: previous left half + current right half
tensor([[[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.9430],
         [ 0.9760],
         [ 0.2182]],

        [[ 2.4564],
         [-0.9331],
         [ 1.8371],
         [-0.4916],
         [-1.1458],
         [-1.0832]],

        [[ 0.6972],
         [ 1.7593],
         [ 0.0283],
         [ 0.0424],
         [ 0.1389],
         [ 0.6271]]])


The overlap pattern means:
- `h_[t] = cat(h[t-1], h[t])` in `[b, s, 2r, d]` format
- `ckv[t] = compressed(h_[t], dim=2)` — each compressed block incorporates its left neighbor

Overlap is applied in both the **outer Compressor** (attention side) and the **inner Compressor** (indexer side).
This strategy smooths block-sparse information and reduces boundary artifacts.

## 6. Official Indexer Implementation

The official implementation has these additional features:
1. Inner Compressor maintains its own `kv_cache` with `index_head_dim=128`
2. RoPE applied to the query `q`
3. Hadamard rotation (`rotate_activation`) + FP4 quantization
4. Causal mask in Prefill mode
5. Offset added to merge window + compressed indices

In [15]:
# Simplified Compressor for Indexer (uses index_head_dim)
class Compressor(nn.Module):
    def __init__(self, args: ModelArgs, compress_ratio: int = 4):
        super().__init__()
        self.ratio = compress_ratio
        self.head_dim = args.index_head_dim
        self.kv_cache = None

    def forward(self, x, start_pos=0, debug=False):
        B, L, D = x.shape
        L_ckv = L // self.ratio
        cKV = torch.randn(B, L_ckv, self.head_dim)
        if self.kv_cache is not None and not debug:
            self.kv_cache[:B, :L_ckv] = cKV
        return cKV

In [16]:
class Indexer(torch.nn.Module):
    """Official V4 Indexer implementation (simplified: no rotation/quantization)."""
    def __init__(self, args: ModelArgs, compress_ratio: int = 4):
        super().__init__()
        self.dim = args.dim
        self.n_heads = args.index_n_heads
        self.head_dim = args.index_head_dim
        self.rope_head_dim = args.rope_head_dim
        self.index_topk = args.index_topk
        self.q_lora_rank = args.q_lora_rank
        self.wq_b = nn.Linear(self.q_lora_rank, self.n_heads * self.head_dim)
        self.weights_proj = nn.Linear(self.dim, self.n_heads)
        self.softmax_scale = self.head_dim ** -0.5
        self.compress_ratio = compress_ratio

        self.compressor = Compressor(args, compress_ratio)
        self.register_buffer("kv_cache", torch.zeros(args.max_batch_size,
                                                         args.max_seq_len // compress_ratio,
                                                         self.head_dim), persistent=False)
        self.freqs_cis = None

    def forward(self, x: torch.Tensor, qr: torch.Tensor, start_pos: int, offset: int):
        bsz, seqlen, _ = x.size()
        ratio = self.compress_ratio
        rd = self.rope_head_dim
        end_pos = start_pos + seqlen

        if self.compressor.kv_cache is None:
            self.compressor.kv_cache = self.kv_cache

        # Step 1: Project query to multi-head
        q = self.wq_b(qr)
        q = q.unflatten(-1, (self.n_heads, self.head_dim))
        # apply_rotary_emb(q[..., -rd:], freqs_cis)  # omitted for simplicity

        # Step 2: Inner compressor updates kv_cache
        self.compressor(x, start_pos)

        # Step 3: Multi-head scoring with weight projection
        weights = self.weights_proj(x) * (self.softmax_scale * self.n_heads ** -0.5)
        index_score = torch.einsum("bshd,btd->bsht", q, self.kv_cache[:bsz, :end_pos // ratio])
        index_score = (index_score.relu_() * weights.unsqueeze(-1)).sum(dim=2)

        # Step 4: Causal mask (Prefill only)
        if start_pos == 0:
            mask = torch.arange(seqlen // ratio).repeat(seqlen, 1) >= \
                   torch.arange(1, seqlen + 1).unsqueeze(1) // ratio
            index_score += torch.where(mask, float("-inf"), 0)

        # Step 5: Top-k selection
        topk_idxs = index_score.topk(min(self.index_topk, end_pos // ratio), dim=-1)[1]

        # Step 6: Offset + invalid mask (Prefill only)
        if start_pos == 0:
            mask = topk_idxs >= torch.arange(1, seqlen + 1).unsqueeze(1) // ratio
            topk_idxs = torch.where(mask, -1, topk_idxs + offset)
        else:
            topk_idxs += offset

        return topk_idxs

indexer = Indexer(args)

In [17]:
print(indexer)  # Show the two main learnable components: wq_b and weights_proj

Indexer(
  (wq_b): Linear(in_features=1024, out_features=8192, bias=True)
  (weights_proj): Linear(in_features=4096, out_features=64, bias=True)
  (compressor): Compressor()
)


In [19]:
# Run Indexer: each query selects top-k=3 compressed KV blocks
# Results show -1 for causally invisible positions, offset=128 for valid ones
seq_len = 100
x = torch.randn(bsz, seq_len, args.dim)
qr = torch.randn(bsz, seq_len, args.q_lora_rank)
topk_idxs = indexer(x, qr, start_pos=0, offset=args.window_size)
print(f"Indexer output (start_pos=0, offset={args.window_size}):")
print(topk_idxs)
print(f"\nInterpretation:")
print(f"  -1 = causally invisible (no compressed KV available)")
print(f"  128+ = valid compressed KV index (offset by window_size={args.window_size})")

Indexer output (start_pos=0, offset=128):
tensor([[[ -1,  -1,  -1],
         [ -1,  -1,  -1],
         [ -1,  -1,  -1],
         [128,  -1,  -1],
         [128,  -1,  -1],
         [128,  -1,  -1],
         [128,  -1,  -1],
         [128, 129,  -1],
         [128, 129,  -1],
         [128, 129,  -1],
         [129, 128,  -1],
         [128, 129, 130],
         [130, 128, 129],
         [130, 128, 129],
         [129, 128, 130],
         [130, 128, 131],
         [128, 131, 129],
         [131, 129, 130],
         [128, 130, 129],
         [129, 132, 128],
         [128, 129, 131],
         [128, 131, 132],
         [129, 132, 131],
         [131, 129, 132],
         [132, 131, 128],
         [128, 130, 133],
         [131, 129, 133],
         [131, 130, 133],
         [134, 128, 130],
         [132, 130, 133],
         [132, 130, 131],
         [134, 131, 132],
         [135, 134, 132],
         [135, 130, 128],
         [135, 129, 133],
         [132, 136, 129],
         [134, 129, 13

# KV Cache Analysis

The Indexer maintains its own `kv_cache` of shape `[B, max_seq_len // ratio, index_head_dim]`.

This is separate from the attention `kv_cache` which has shape `[B, window_size + max_seq_len // ratio, head_dim]`.

Key observations:
- Indexer's `kv_cache` uses `index_head_dim=128` (vs attention's `head_dim=512`)
- Indexer's cache grows incrementally with sequence length
- The inner Compressor also maintains `kv_state` and `score_state` for incremental decoding